# Model lineal de predicció demogràfica · Barcelona 1997–2026

+ **Durada estimada:** 2 hores
+ **Prerequisit:** exercici1.ipynb
+ **Eines:** Python 3 · pandas · numpy · matplotlib

---

## Pregunta de recerca

A partir de les dades de la gràfica *Distribució de la població per lloc de
naixement · 1997–2026*, construeix un model lineal **implementat des de zero,
amb gradient descendent**, que permeti respondre:

> **Si no canvien les tendències, quan superarà la població nascuda a la «Resta
> del món» la suma de la població nascuda a Barcelona, la Resta de Catalunya i
> la Resta d'Espanya?**

Recordatori dels grups del gràfic (`LLOC_NAIX`):

| Codi | Descripció |
|------|------------|
| 1 | Barcelona ciutat |
| 2 | Resta de Catalunya |
| 3 | Resta d'Espanya |
| 4 | Resta de la Unió Europea |
| 5 | Resta del món |
| 6 | No consta |

> **Nota:** Les cel·les marcades amb 🔧 contenen codi que has d'escriure o completar.
> Les marcades amb ✅ ja estan resoltes i serveixen de guia.

## Configuració inicial

In [ ]:
# ✅
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob
import os

BASE          = "./data"
PATH_POBLACIO = os.path.join(BASE, "poblacio_lloc_naix")
PATH_DIMS     = os.path.join(BASE, "pad_dimensions.csv")

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.35,
    "grid.linestyle": "--",
})
print("\u2713 Configuraci\u00f3 carregada")

In [ ]:
# ✅ Càrrega de tota la sèrie temporal
def netejar_poblacio(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    df["Valor"] = pd.to_numeric(df["Valor"], errors="coerce")
    df = df[df["LLOC_NAIX"].isin(range(1, 7))].copy()
    df["Any"] = pd.to_datetime(df["Data_Referencia"]).dt.year
    return df

fitxers  = sorted(glob.glob(os.path.join(PATH_POBLACIO, "*.csv")))
df_serie = pd.concat([netejar_poblacio(f) for f in fitxers], ignore_index=True)

serie_anual = (
    df_serie
    .groupby(["Any", "LLOC_NAIX"], as_index=False)["Valor"]
    .sum()
    .rename(columns={"Valor": "Total"})
)
print(f"\u2713 {df_serie['Any'].nunique()} anys carregats "
      f"({df_serie['Any'].min()}\u2013{df_serie['Any'].max()})")

---
## Part 1 — Dos grups de comparació

| Grup | Codis | Significat |
|------|-------|------------|
| **local** | 1, 2, 3 | Nascuts a Barcelona, Catalunya o Espanya |
| **exterior** | 4, 5 | Nascuts fora d'Espanya (UE, resta del món) |

Calcula per a cada any la població total de cada grup i la seva diferència
(`local − exterior`). Quan la diferència arribi a zero, els dos grups s'hauran
igualat.

In [ ]:
# 🔧 Crea df_comparacio amb columnes: Any | local | exterior | diferencia
#
# Pistes:
#   1. Afegeix una columna "Grup" a serie_anual ("local" per als codis 1,2,3;
#      "exterior" per als codis 4,5). Pots usar np.where(...).
#   2. Agrupa per ["Any", "Grup"] i suma Total.
#   3. Fes un pivot per tenir local i exterior com a columnes.
#   4. Calcula diferencia = local − exterior.

df_comparacio = ...  # 🔧

print(df_comparacio.tail(5).to_string(index=False))

---
## Part 2 — Visualització de les tendències

In [ ]:
# 🔧 Gràfic de 2 línies: grup "local" i grup "exterior" 1997–2026
#
# Requisits:
#   - Dues línies amb colors diferenciats
#   - Títol, etiquetes d'eixos i llegenda
#   - Eix Y en milers (FuncFormatter)
#   - Anotació del valor final (2026) a la dreta de cada línia

fig, ax = plt.subplots(figsize=(11, 5))
# 🔧 Escriu el codi aquí

plt.tight_layout()
plt.show()

---
## Part 3 — Regressió lineal i gradient descendent: conceptes

### El model

Volem ajustar una recta que descrigui com evoluciona la diferència
(`local − exterior`) al llarg del temps:

$$\hat{y} = m \cdot t + b$$

on $t$ és l'any, $\hat{y}$ és la diferència predita, $m$ és el
**pendent** i $b$ el **terme independent**.

### La funció de cost (MSE)

Per mesurar com de bé s'ajusta la recta usem l'**error quadràtic
mitjà** (*mean squared error*):

$$J(m, b) = \frac{1}{n} \sum_{i=1}^{n} \left(y_i - \hat{y}_i\right)^2
          = \frac{1}{n} \sum_{i=1}^{n} \left(y_i - (m \cdot t_i + b)\right)^2$$

L'objectiu és trobar els valors de $m$ i $b$ que **minimitzen** $J$.

### Gradient descendent

El **gradient descendent** actualitza $m$ i $b$ iterativament en la
direcció que redueix el cost:

$$m \leftarrow m - \alpha \cdot \frac{\partial J}{\partial m}
\qquad
b \leftarrow b - \alpha \cdot \frac{\partial J}{\partial b}$$

on $\alpha$ és la **taxa d'aprenentatge** (*learning rate*). 

Derivant $J$:

$$\frac{\partial J}{\partial m} = \frac{-2}{n}
  \sum_{i=1}^{n} t_i \cdot \left(y_i - (m \cdot t_i + b)\right)$$

$$\frac{\partial J}{\partial b} = \frac{-2}{n}
  \sum_{i=1}^{n} \left(y_i - (m \cdot t_i + b)\right)$$

### Per què normalitzar les dades?

Els anys $t$ van de 1997 a 2026 (valors grans) i les diferències $d$ de
població son de l'ordre de centenars de milers. Quan les escales
dels inputs i outputs son molt diferents, els gradients tenen
magnituds molt dispars i el descens és lent o inestable.

**Solució: estandardització** (*z-score*):

$$t_{\text{norm}} = \frac{t - \mu_t}{\sigma_t}
\qquad
d_{\text{norm}} = \frac{d - \mu_d}{\sigma_d}$$

Amb dades estandarditzades, una taxa d'aprenentatge $\alpha \in [0.01, 0.5]$ sol funcionar bé.

---
## Part 4 — Implementació del gradient descendent

Implementa el gradient descendent en quatre passos: normalització,
funció de cost, gradients i bucle d'entrenament.

In [ ]:
# 🔧 4.1 · Estandarditza els anys i les diferències
#
# t_norm = (anys - anys.mean()) / anys.std()
# d_norm = (difs - difs.mean()) / difs.std()
#
# Guarda també les estadístiques originals (les necessitaràs per
# tornar a l'escala original més endavant).

anys = df_comparacio["Any"].values.astype(float)
difs = df_comparacio["diferencia"].values

t_mean, t_std = ...  # 🔧
d_mean, d_std = ...  # 🔧

t_norm = ...  # 🔧
d_norm = ...  # 🔧

print(f"t_norm: mean = {t_norm.mean():.6f}  std = {t_norm.std():.4f}")
print(f"d_norm: mean = {d_norm.mean():.6f}  std = {d_norm.std():.4f}")
# Hauria de sortir mean ≈ 0 i std ≈ 1 per a les dues variables

In [ ]:
# 🔧 4.2 · Funció de cost: MSE
#
# J(m, b) = (1/n) * sum((y_i - (m*t_i + b))^2)

def cost_mse(m, b, t, y):
    """
    Calcula l'error quadràtic mitjà del model y_pred = m*t + b.

    Paràmetres:
        m, b : float  — pendent i terme independent actuals
        t    : array  — anys (normalitzats)
        y    : array  — diferències (normalitzades)
    Retorna:
        float — valor del MSE
    """
    # 🔧 Escriu el codi aquí
    ...

# Comprova: amb m=0, b=0 i dades estandarditzades el cost hauria de ser ≈ 1.0
print(f"Cost inicial (m=0, b=0): {cost_mse(0, 0, t_norm, d_norm):.6f}")

In [ ]:
# 🔧 4.3 · Càlcul dels gradients
#
# dJ/dm = (-2/n) * sum( t_i * (y_i - (m*t_i + b)) )
# dJ/db = (-2/n) * sum(        y_i - (m*t_i + b)  )

def compute_gradients(m, b, t, y):
    """
    Calcula les derivades parcials del MSE respecte a m i b.

    Retorna:
        (dL_dm, dL_db) : tuple de floats
    """
    n = len(t)
    # 🔧 Escriu el codi aquí
    ...

# Comprova: el gradient ha de ser no-zero per a m=0, b=0
dm, db = compute_gradients(0, 0, t_norm, d_norm)
print(f"Gradient inicial: dJ/dm = {dm:.4f},  dJ/db = {db:.4f}")

In [ ]:
# 🔧 4.4 · Bucle de gradient descendent
#
# Per a cada iteració:
#   1. Calcula el cost i guarda'l a history.
#   2. Calcula els gradients amb compute_gradients.
#   3. Actualitza: m ← m - lr * dL_dm
#                  b ← b - lr * dL_db
#
# Tria lr i n_iter de manera que el cost final convergeixi
# (la corba d'aprenentatge s'ha d'aplanar).

lr     = ...  # 🔧  prova valors entre 0.01 i 0.5
n_iter = ...  # 🔧  prova entre 200 i 2000

m_norm, b_norm = 0.0, 0.0
history = []

for i in range(n_iter):
    # 🔧 Escriu el cos del bucle aquí
    ...

print(f"Par\u00e0metres finals (espai norm.): m = {m_norm:.4f}, b = {b_norm:.4f}")
print(f"Cost inicial: {history[0]:.6f}  |  Cost final: {history[-1]:.6f}")

In [ ]:
# 🔧 4.5 · Visualitza la corba d'aprenentatge (MSE vs. iteració)
#
# La corba ha de mostrar un descens ràpid al principi que s'aplanï.
# Si oscil·la o divergeix, redueix lr.

fig, ax = plt.subplots(figsize=(9, 4))
# 🔧 Escriu el codi aquí

plt.tight_layout()
plt.show()

In [ ]:
# 🔧 4.6 · Converteix els paràmetres de l'espai normalitzat a l'original
#
# El model normalitzat és:  d_norm = m_norm * t_norm + b_norm
# Substituint t_norm i d_norm:
#
#   (d - d_mean)/d_std = m_norm * (t - t_mean)/t_std + b_norm
#
# Aïlla d per obtenir:
#
#   d = m_orig * t + b_orig
#
# on:
#   m_orig = m_norm * d_std / t_std
#   b_orig = d_std * b_norm + d_mean - m_orig * t_mean

m_orig = ...  # 🔧
b_orig = ...  # 🔧

print(f"Model (escala original):")
print(f"  difer\u00e8ncia = {m_orig:.0f} \u00d7 any  +  {b_orig:.0f}")
print(f"  Pendent: {m_orig:.0f} persones/any")

In [ ]:
# ✅ Validació: compara el resultat de GD amb np.polyfit
coefs_ref = np.polyfit(anys, difs, 1)

print(f"{'M\u00e8tode':<20} {'m (pendent)':>14} {'b (intercept)':>15}")
print("-" * 52)
print(f"{'Gradient descendent':<20} {m_orig:>14.1f} {b_orig:>15.0f}")
print(f"{'np.polyfit (ref.)':<20} {coefs_ref[0]:>14.1f} {coefs_ref[1]:>15.0f}")
print()
err_m = abs(m_orig - coefs_ref[0]) / abs(coefs_ref[0]) * 100
err_b = abs(b_orig - coefs_ref[1]) / abs(coefs_ref[1]) * 100
print(f"Error relatiu:  m = {err_m:.2f}%   b = {err_b:.2f}%")
print("Si l'error < 1%, el gradient descendent ha convergit correctament.")

---
## Part 5 — Punt de creuament

La diferència comença positiva i, si la tendència és lineal, arribarà a zero
en algun any futur. Aquell any és la resposta a la nostra pregunta.

Resol analíticament:

$$m_{\text{orig}} \cdot t^{*} + b_{\text{orig}} = 0
\quad \Longrightarrow \quad
t^{*} = -\frac{b_{\text{orig}}}{m_{\text{orig}}}$$

In [ ]:
# 🔧 Calcula l'any de creuament usant m_orig i b_orig

any_creuament = ...  # 🔧

print(f"Any de creuament: {any_creuament:.2f}")
print(f"\u2192 Cap a l'any {int(np.ceil(any_creuament))}, la 'Resta del m\u00f3n' superarà BCN+Cat+Esp")

---
## Part 6 — Visualització final amb projecció

Crea un gràfic que mostri:
- Els punts de la **diferència observada** (1997–2026).
- La **recta del model** extrapolada fins al punt de creuament (línia discontínua).
- La **línia zero** i el **punt de creuament** anotat.

In [ ]:
# 🔧 Gràfic de la diferència + model + projecció
#
# Defineix el model com a funció de t:
#   model_orig = lambda t: m_orig * t + b_orig
#
# Pistes:
#   anys_proj = np.arange(1997, int(any_creuament) + 6)
#   ax.axhline(0, ...)                 → línia zero
#   ax.axvline(any_creuament, ...)     → any de creuament
#   ax.scatter([any_creuament], [0])   → punt de creuament
#   ax.annotate(...)                   → etiqueta l'any

model_orig = lambda t: m_orig * t + b_orig

fig, ax = plt.subplots(figsize=(12, 5.5))
# 🔧 Escriu el codi aquí

plt.tight_layout()
plt.savefig("grafic_projeccio.png", bbox_inches="tight", dpi=150)
plt.show()

---
## Part 7 — Interpretació crítica

Respon les preguntes següents en les cel·les Markdown (3-5 frases per apartat).

**a) Quan superarà la «Resta del món» els grups locals? Quin és el grau de
confiança d'aquesta predicció?**

*🔧 Escriu la teva resposta aquí.*

**b) Quina influència té la tria de `lr` (learning rate) i `n_iter` sobre
el resultat? Qu passa si `lr` és massa gran? I massa petit?**

*🔧 Escriu la teva resposta aquí.*

**c) Un model lineal assumeix un ritme de canvi constant. Observes algun
període on la tendència va ser significativament diferent (crisi 2008,
pandèmia 2020)? Com afecta aixó a la fiabilitat de la predicció?**

*🔧 Escriu la teva resposta aquí.*